# Projeto final do MBA — auditoria inicial de queimadas

Objetivo: verificar integralmente os quatro CSVs fornecidos antes da preparação e modelagem. Este notebook não representa a Entrega 1 completa. Não há alvo rotulado nem modelo treinado. Os dados brutos são preservados.

O relatório técnico foi produzido por `auditar_bases_queimadas.py`, com leitura em blocos de 100 mil linhas. Duplicatas são estimadas por hash de 64 bits; validar igualdade antes de qualquer exclusão.

In [ ]:
from pathlib import Path
import os, sys
_raiz = Path.cwd()
while not (_raiz / 'preparar_eda_queimadas.py').exists() and _raiz.parent != _raiz:
    _raiz = _raiz.parent
os.chdir(_raiz)
sys.path.insert(0, str(_raiz))
from pathlib import Path
import json
import pandas as pd
from auditar_bases_queimadas import audit

ROOT = Path.cwd()
REPROCESSAR = False  # True refaz a varredura integral dos quatro CSVs.
perfil = ROOT / 'auditoria_queimadas' / 'perfil_bases.json'
if REPROCESSAR or not perfil.exists():
    resultado = audit(ROOT)
else:
    resultado = json.loads(perfil.read_text(encoding='utf-8'))
print('Bases auditadas:', len(resultado['bases']))

## 1. Inventário e integridade

As verificações abaixo conferem a presença, o tamanho e o esquema registrados na auditoria. O SHA-256 no manifesto permite verificar o conteúdo; o tamanho isolado não substitui esse hash.

In [ ]:
inventario = pd.DataFrame([{'arquivo': b['arquivo'], 'linhas': b['linhas'], 'colunas': len(b['colunas']), 'bytes': b['bytes'], 'separador': b['separador'], 'repeticoes_hash64': b['duplicatas_hash64']} for b in resultado['bases']])
print(inventario.to_string(index=False))
for b in resultado['bases']:
    caminho = ROOT / 'dados_brutos_queimadas' / b['arquivo']
    assert caminho.is_file() and caminho.stat().st_size == b['bytes']
    assert b['linhas'] > 0
print('Presenca e tamanho conferidos nos quatro arquivos.')

## 2. Cobertura temporal

A interseção das datas extremas é uma condição necessária para a junção temporal. Não demonstra cobertura completa por município/dia. Dados posteriores ao fim da série de queimadas não devem ser rotulados automaticamente como ausência de incêndio.

In [ ]:
for b in resultado['bases']:
    anos = sorted(b['categorias'].get('ano', {}))
    print(b['arquivo'], 'anos:', (anos[0], anos[-1]) if anos else 'cadastro', 'datas:', b['datas'])
fogo = next(b for b in resultado['bases'] if b['arquivo'] == 'queimadas_2024_2025.csv')
meteo = next(b for b in resultado['bases'] if b['arquivo'] == 'BDMEP-2025.csv')
inicio = max(fogo['datas']['data_hora']['min'], meteo['datas']['data']['min'])
fim = min(fogo['datas']['data_hora']['max'], meteo['datas']['data']['max'])
print('Intersecao observada:', inicio, 'a', fim, '| dias:', max(0, (pd.Timestamp(fim)-pd.Timestamp(inicio)).days+1))

## 3. Ausências, repetições e ligações

Campos vazios são contados separadamente de valores negativos ou códigos especiais. Temperaturas negativas podem ser válidas. Códigos -999 em risco e dias sem chuva precisam de confirmação documental e não devem ser tratados como medidas normais.

In [ ]:
for b in resultado['bases']:
    vazios = {c: n for c, n in b['vazios'].items() if n}
    print('\n' + b['arquivo'])
    print('Vazios:', vazios)
    if 'chave_candidata' in b:
        print('Chave candidata:', b['chave_candidata'], '| repeticoes hash:', b['repeticoes_chave_hash64'])
print('\nLigacoes:', json.dumps(resultado['ligacoes'], ensure_ascii=False, indent=2))

## 4. Dicionário: divergências observadas

A legenda recebida do MapBiomas contém somente classes de 1 a 7. Os códigos abaixo exigem uma legenda da coleção efetivamente exportada. Nenhum significado foi atribuído por suposição.

In [ ]:
mapa = next(b for b in resultado['bases'] if b['arquivo'] == 'MapBiomas.csv')
classes = sorted(mapa['categorias']['id_classe'], key=int)
print('Classes observadas:', len(classes), classes)
print('Risco de fogo:', fogo['numericos']['risco_fogo'])
print('Satélites/algoritmos:', len(fogo['categorias']['satelite']))

## 5. Decisões pendentes antes do EDA orientado e modelagem

1. Confirmar se baixo/médio/alto representa faixas do índice `risco_fogo` ou risco de ocorrência futura, e documentar horizonte e momento da previsão.
2. Confirmar se existe arquivo de queimadas atualizado ou meteorologia de 2024; com os arquivos atuais, a janela comum observada tem 43 dias. Não substituir as bases fornecidas sem registrar a mudança.
3. Obter a coleção e legenda do MapBiomas e conferir as unidades meteorológicas do dicionário.
4. Confirmar duplicatas exatas e resolver divergências por estação/data/hora antes de agregar.
5. Definir a população município/dia e a cobertura de detecção antes de criar exemplos sem foco.
6. Separar períodos de treino/validação/teste e ajustar transformações somente no treino.

KPIs candidatos, condicionados à definição do alvo: recall da classe de alto risco, F1 macro, matriz de confusão e custo operacional de falsos alertas e eventos perdidos. O impacto financeiro exigirá premissas explícitas; nenhum valor foi estimado nesta auditoria.

A Entrega 1 ainda exige EDA interpretada, teste de hipótese e base consolidada. A definição do alvo precede conclusões sobre correlações e sucesso do modelo.